# 01 — Data Ingestion
Parses VCF / VCF.bgz files into a DataFrame using `src/ingest.py` and saves to parquet for downstream notebooks.

In [ ]:
import sys, os
from pathlib import Path

import pandas as pd

ROOT = Path(os.getcwd()).parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from ingest import parse_vcf_to_dataframe

In [ ]:
# ── Configure ──────────────────────────────────────────────────────────────
VCF_FILES = [
    ROOT / 'data' / 'raw' / 'example.vcf.bgz',   # ← replace / add paths
]

# Set to an integer (e.g. 10_000) for a quick test run; None = full file
MAX_ROWS: int | None = None

OUT_DIR = ROOT / 'data' / 'raw'
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Parse ──────────────────────────────────────────────────────────────────
frames = []

for vcf_path in VCF_FILES:
    vcf_path = Path(vcf_path)
    if not vcf_path.exists():
        print(f'[SKIP] not found: {vcf_path}')
        continue

    print(f'Parsing {vcf_path.name} ...', end=' ', flush=True)
    df = parse_vcf_to_dataframe(str(vcf_path), max_rows=MAX_ROWS)
    df['source_file'] = vcf_path.name
    frames.append(df)
    print(f'{len(df):,} variants')

if not frames:
    raise FileNotFoundError('No VCF files found. Check VCF_FILES above.')

variants = pd.concat(frames, ignore_index=True)
print(f'\nTotal variants: {len(variants):,}')
variants.head()

In [ ]:
# ── Save ───────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'variants.parquet'
variants.to_parquet(out_path, index=False, engine='pyarrow')
print(f'Saved → {out_path}')